# 🚀 تدريب موديل Qwen2.5-3B لمنع اللهلوسة والتوافق التام مع نظام Nebula RAG

هذا الدفتر مخصص لتدريب سريع (Fine-Tuning) باستخدام تقنية **QLoRA (4-bit)** على كارت شاشة مجاني من Google Colab (T4 GPU).

### أهداف التدريب:
1. **إلزام النموذج بسياق RAG**: الإجابة فقط من واقع الذكريات المسترجعة.
2. **منع اللهلوسة تماماً (Anti-Hallucination)**: تدريب النموذج على الاعتراف الصريح بعدم المعرفة عند غياب المعلومة بدلاً من اختلاق تفاصيل كاذبة.
3. **تثبيت اللهجة المصرية وشخصية نيبولا** والوسوم التعبيرية `[happy]`, `[wave]`, `[like]`.

In [ ]:
# 1. تثبيت الحزم اللازمة
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# 2. التحقق من كارت الشاشة (GPU)
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ الرجاء تغيير Runtime في Colab إلى T4 GPU!")

In [ ]:
# 3. رفع ملف الداتاسيت الديناميكي (train_chatml.jsonl)
# تم توليد هذا الملف ديناميكياً من قاعدة بيانات نيبولا عبر: python scripts/generate_rag_dataset.py
from google.colab import files
import os

if not os.path.exists("train_chatml.jsonl"):
    print("📂 يرجى رفع ملف train_chatml.jsonl المولد ديناميكياً من مجلد data/rag_finetune/:")
    uploaded = files.upload()
else:
    print("✅ ملف train_chatml.jsonl متوفر بالفعل!")

In [ ]:
# 4. تحميل الموديل والـ Tokenizer بتقنية 4-bit QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. بدء التدريب على الداتاسيت الديناميكية
dataset = load_dataset("json", data_files="train_chatml.jsonl")

training_args = TrainingArguments(
    output_dir="./nebula_rag_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    dataset_text_field="messages",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
model.save_pretrained("./nebula_rag_adapter")
tokenizer.save_pretrained("./nebula_rag_adapter")
print("🎉 تم حفظ أوزان LoRA بنجاح!")

In [ ]:
# 6. تجربة واختبار النموذج بعد التدريب مباشرة
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# قراءة أول عينة اختبار من الداتاسيت لفحص السلوك
test_prompt = "<|im_start|>user\nمعلومات الذاكرة الموثوقة (RAG):\n- المستخدم يفضل القهوة بدون سكر\n\nالسؤال: ما هو الرقم السري لخزنة المكتب؟<|im_end|>\n<|im_start|>assistant\n"
output = pipe(test_prompt, max_new_tokens=60, temperature=0.1, do_sample=True)
print("رد النموذج:")
print(output[0]["generated_text"].split("<|im_start|>assistant\n")[-1])